In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Visualisation des résultats avec GradCAM\n",
    "\n",
    "Ce notebook permet de visualiser les résultats du modèle ResNet50 en utilisant la technique GradCAM pour identifier les zones qui contribuent le plus à la prédiction."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import os\n",
    "import sys\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import tensorflow as tf\n",
    "from tensorflow.keras.models import load_model\n",
    "import cv2\n",
    "\n",
    "# Ajouter le répertoire racine au chemin pour pouvoir importer les modules\n",
    "sys.path.append(os.path.abspath('../'))\n",
    "\n",
    "from src.data.data_loader import DataLoader\n",
    "from src.model.resnet50_model import ResNet50Model\n",
    "from src.visualization.gradcam import GradCAM"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Chargement du modèle entraîné"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Charger le modèle entraîné\n",
    "MODEL_PATH = '../models/resnet50_finetune.h5'\n",
    "model = load_model(MODEL_PATH)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Préparation des images de test"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Charger les données de test\n",
    "DATA_DIR = '../data'\n",
    "PROCESSED_DATA_DIR = os.path.join(DATA_DIR, 'processed')\n",
    "\n",
    "X_test = np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy'))\n",
    "y_test = np.load(os.path.join(PROCESSED_DATA_DIR, 'y_test.npy'))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Initialisation de GradCAM"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Initialiser GradCAM avec le modèle ResNet50\n",
    "# Pour ResNet50, la couche de convolution finale est souvent 'conv5_block3_out'\n",
    "gradcam = GradCAM(model, \"conv5_block3_out\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Visualisation des résultats sur plusieurs images"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "def visualize_predictions(images, labels, indices, gradcam):\n",
    "    fig, axes = plt.subplots(len(indices), 3, figsize=(15, 5*len(indices)))\n",
    "    \n",
    "    for i, idx in enumerate(indices):\n",
    "        img = images[idx]\n",
    "        true_label = \"Pneumonie\" if labels[idx] == 1 else \"Normal\"\n",
    "        \n",
    "        # Faire une prédiction\n",
    "        img_batch = np.expand_dims(img, axis=0)\n",
    "        prediction = model.predict(img_batch)[0][0]\n",
    "        pred_label = \"Pneumonie\" if prediction > 0.5 else \"Normal\"\n",
    "        \n",
    "        # Générer la carte de chaleur GradCAM\n",
    "        heatmap = gradcam.compute_heatmap(img_batch, pred_class=0 if prediction <= 0.5 else 1)\n",
    "        cam_image = gradcam.overlay_heatmap(img, heatmap, alpha=0.5)\n",
    "        \n",
    "        # Afficher l'image originale\n",
    "        axes[i, 0].imshow(img)\n",
    "        axes[i, 0].set_title(f\"Original - Vrai: {true_label}\")\n",
    "        axes[i, 0].axis('off')\n",
    "        \n",
    "        # Afficher la carte de chaleur seule\n",
    "        axes[i, 1].imshow(heatmap)\n",
    "        axes[i, 1].set_title(\"Carte de chaleur GradCAM\")\n",
    "        axes[i, 1].axis('off')\n",
    "        \n",
    "        # Afficher l'image avec la carte de chaleur superposée\n",
    "        axes[i, 2].imshow(cam_image)\n",
    "        axes[i, 2].set_title(f\"Prédiction: {pred_label} ({prediction:.2f})\")\n",
    "        axes[i, 2].axis('off')\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Sélectionner des indices d'images pour visualisation\n",
    "# Inclure quelques cas positifs et négatifs\n",
    "normal_indices = np.where(y_test == 0)[0][:3]  # 3 cas normaux\n",
    "pneumonia_indices = np.where(y_test == 1)[0][:3]  # 3 cas de pneumonie\n",
    "\n",
    "# Visualiser les cas normaux\n",
    "visualize_predictions(X_test, y_test, normal_indices, gradcam)\n",
    "\n",
    "# Visualiser les cas de pneumonie\n",
    "visualize_predictions(X_test, y_test, pneumonia_indices, gradcam)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Analyse des cas difficiles"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Faire des prédictions sur l'ensemble du jeu de test\n",
    "predictions = model.predict(X_test)\n",
    "predicted_classes = (predictions > 0.5).astype(int).flatten()\n",
    "\n",
    "# Identifier les cas incorrectement classés\n",
    "incorrect_indices = np.where(predicted_classes != y_test)[0]\n",
    "print(f\"Nombre de cas incorrectement classés: {len(incorrect_indices)}\")\n",
    "\n",
    "# Visualiser quelques cas incorrectement classés\n",
    "if len(incorrect_indices) > 0:\n",
    "    # Sélectionner jusqu'à 3 cas incorrects à visualiser\n",
    "    sample_indices = incorrect_indices[:min(3, len(incorrect_indices))]\n",
    "    visualize_predictions(X_test, y_test, sample_indices, gradcam)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Conclusion\n",
    "\n",
    "La visualisation GradCAM nous permet de mieux comprendre comment le modèle prend ses décisions. Les zones rouges/oranges sur la carte de chaleur indiquent les régions de l'image qui ont le plus contribué à la classification.\n",
    "\n",
    "Pour une pneumonie, le modèle devrait se concentrer sur les zones pulmonaires qui présentent des opacités ou des infiltrats. Pour les cas normaux, l'attention devrait être plus diffuse ou se concentrer sur les structures normales des poumons.\n",
    "\n",
    "Cette visualisation est particulièrement utile pour:\n",
    "1. Valider que le modèle se concentre sur les zones anatomiquement pertinentes\n",
    "2. Identifier les cas où le modèle pourrait utiliser des artefacts ou des caractéristiques non pertinentes\n",
    "3. Améliorer la confiance des radiologues dans les prédictions du modèle"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}